Casualty Data Preparation

Project Objective

This notebook prepares the official UK Department for Transport casualty dataset for the machine learning project.

The main objectives are:

- Load the casualty dataset.
- Keep only casualties involved in collisions from 2010 onwards.
- Explore the dataset structure.
- Assess data quality.
- Remove unnecessary variables.
- Prepare a clean dataset for the data integration stage.

Import Libraries

In [1]:
import pandas as pd
import numpy as np

Load Dataset

Load the cleaned collision dataset together with the official casualty dataset.

Only casualties associated with collisions from 2010 onwards are retained to ensure consistency across the project.

In [2]:
df = pd.read_csv('/content/drive/MyDrive/UK-Accident-Prediction/dft-road-casualty-statistics-casualty-1979-latest-published-year.csv',
                 nrows=5,
                 dtype={'collision_index': 'string', 'collision_ref_no': 'string'})

In [3]:
df.head()

,collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,...,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding
0,197905H200700,1979,05H200700,1,1,1,1,20,4,3,...,0,-1,109,-1,-1,-1,-1,NaN,NaN,-1
1,197952FKE0696,1979,52FKE0696,1,1,2,2,18,4,3,...,0,-1,109,-1,-1,-1,-1,NaN,NaN,-1
2,197997NC01901,1979,97NC01901,2,1,2,1,65,9,3,...,0,-1,109,-1,-1,-1,-1,NaN,NaN,-1
3,1979316110883,1979,316110883,1,1,2,1,52,8,3,...,0,-1,109,-1,-1,-1,-1,NaN,NaN,-1
4,197901F9MCD21,1979,01F9MCD21,1,1,2,1,87,11,3,...,1,-1,11,-1,-1,-1,-1,NaN,NaN,-1


In [4]:
collisions = set(
    pd.read_csv('/content/drive/MyDrive/UK-Accident-Prediction/collision_clean.csv',
                usecols=['collision_index'],
                dtype={'collision_index': 'string'})['collision_index']
)

In [6]:
chunks = pd.read_csv(
    '/content/drive/MyDrive/UK-Accident-Prediction/dft-road-casualty-statistics-casualty-1979-latest-published-year.csv',
    chunksize=100000,
    dtype={'collision_index': 'string', 'collision_ref_no': 'string'})

In [7]:
df = pd.concat([
    chunk.loc[chunk['collision_index'].isin(collisions)]
    for chunk in chunks],
               ignore_index=True)

/tmp/ipykernel_2305/2347136257.py:3: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks],
/tmp/ipykernel_2305/2347136257.py:3: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks],
/tmp/ipykernel_2305/2347136257.py:3: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks],
/tmp/ipykernel_2305/2347136257.py:3: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks],
/tmp/ipykernel_2305/2347136257.py:3: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks],
/tmp/ipykernel_2305/2347136257.py:3: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks],
/tmp/ipykernel_2305/23471362

In [8]:
del collisions

In [9]:
df.shape

(2479311, 23)

In [10]:
df.memory_usage(deep=True).sum()/1024**2

np.float64(796.4173498153687)

Exploration and Data Cleaning

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2479311 entries, 0 to 2479310
Data columns (total 23 columns):
 #   Column                              Dtype  
---  ------                              -----  
 0   collision_index                     string 
 1   collision_year                      int64  
 2   collision_ref_no                    string 
 3   vehicle_reference                   int64  
 4   casualty_reference                  int64  
 5   casualty_class                      int64  
 6   sex_of_casualty                     int64  
 7   age_of_casualty                     int64  
 8   age_band_of_casualty                int64  
 9   casualty_severity                   int64  
 10  pedestrian_location                 int64  
 11  pedestrian_movement                 int64  
 12  car_passenger                       int64  
 13  bus_or_coach_passenger              int64  
 14  pedestrian_road_maintenance_worker  int64  
 15  casualty_type                       int64  
 16  

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
df.duplicated(subset=['collision_index', 'casualty_reference']).sum()

np.int64(9)

In [14]:
missing = df.isna().sum()

In [15]:
missing[missing>0].sort_values(ascending=False)

,0


In [16]:
duplicates = df[df.duplicated(subset=['collision_index', 'casualty_reference'], keep=False)].sort_values(['collision_index', 'casualty_reference'])

In [17]:
duplicates

,collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,...,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding
584377,201104CK11089,2011,04CK11089,2,3,2,2,28,6,3,...,0,0,9,-1,E01007316,-1,0,0.07657,0.92343,-1
1494624,201104CK11089,2011,04CK11089,1,3,2,2,59,9,3,...,0,0,9,-1,E01006985,-1,0,0.11837,0.88163,-1
1036912,201104FJ11013,2011,04FJ11013,2,4,1,1,56,9,3,...,0,0,9,-1,E01025375,-1,0,0.05094,0.94906,-1
1078828,201104FJ11013,2011,04FJ11013,1,4,2,2,17,4,3,...,0,0,9,-1,E01025369,-1,0,0.08062,0.91938,-1
711728,201204CL12218,2012,04CL12218,2,1,1,2,39,7,3,...,0,0,9,-1,E01025290,-1,0,0.16800,0.83200,-1
2033378,201204CL12218,2012,04CL12218,1,1,1,1,24,5,3,...,0,0,9,-1,E01025422,-1,0,0.13537,0.86463,-1
95607,201304CJ13162,2013,04CJ13162,1,3,2,1,13,3,3,...,0,0,9,-1,E01025483,-1,0,0.07955,0.92045,-1
1375537,201304CJ13162,2013,04CJ13162,2,3,1,1,51,8,3,...,0,0,9,-1,E01006712,-1,0,0.07313,0.92687,-1
1664039,201304EA13001,2013,04EA13001,2,3,2,2,29,6,3,...,0,0,9,-1,E01012612,-1,0,0.05086,0.94914,-1
1953032,201304EA13001,2013,04EA13001,1,3,2,1,13,3,3,...,0,0,9,-1,E01025065,-1,0,0.04849,0.95151,-1


In [18]:
df.drop(
    columns=[
        'collision_ref_no',
        'casualty_severity',
        'enhanced_casualty_severity',
        'casualty_injury_based',
        'casualty_adjusted_severity_serious',
        'casualty_adjusted_severity_slight'
    ],inplace=True
)

In [19]:
df['age_of_casualty'] = df['age_of_casualty'].replace(-1,float('nan'))

In [20]:
df.shape

(2479311, 17)

In [21]:
df.duplicated(subset=[
    'collision_index',
    'vehicle_reference',
    'casualty_reference'
]).sum()

np.int64(0)

Save Clean Dataset

Export the cleaned casualty dataset.

The resulting dataset will be combined with the collision and vehicle datasets during the data integration stage.

In [22]:
df.to_csv('/content/drive/MyDrive/UK-Accident-Prediction/casualty_clean.csv')

## Preparation Summary

- Retained casualties associated with the selected collisions.
- Removed the redundant collision reference and severity-related columns.
- Converted missing age codes (-1) to NaN.
- Retained records with repeated collision and casualty references because their vehicle references and personal attributes differ.
- Exported the prepared dataset for collision-level aggregation.

In [23]:
print('Casualty Dataset Summary')
print('='*50)
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(f'Missing Values: {df.isnull().sum().sum()}')
print(f'Duplicate Rows: {df.duplicated().sum()}')

Casualty Dataset Summary
Rows: 2479311
Columns: 17
Missing Values: 44872
Duplicate Rows: 0
